# Phase 3 RR: LoRA, LoRMA, full fine-tuning and distillation on my phase 1 gqa_swa model

Attach the phase 1 output (`gqa_swa_best.pt` and `bpe_legal_16k_v2.pkl`) and add the `HF` secret.

In [ ]:
import os, sys, json, math, random, time, zipfile, glob, re, pickle, gc
from collections import Counter, defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint as grad_ckpt
from tqdm.auto import tqdm
from huggingface_hub import login, hf_hub_download
from sklearn.metrics import f1_score, precision_score, recall_score
from types import SimpleNamespace

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
    print("Logged in.")
else:
    print("WARNING: HF_TOKEN not found, public datasets will download without auth.")


## Config

In [ ]:
# every sentence is cut to MAX_LEN tokens (prev + target + next)
MAX_LEN = 256
TRAIN_BATCH, EVAL_BATCH = 16, 32
EPOCHS = 3
LR = 5e-4
FULL_LR = 2e-5
WEIGHT_DECAY = 0.01
# the blocks are checkpointed while training, so full ft fits on the gpu
USE_GC = True
SEED = 42

RANK, ALPHA = 8, 16.0
LORMA_RANK, LORMA_ALPHA = 16, 32.0
STUDENT_RANK, STUDENT_ALPHA = 4, 8.0
KD_TEMP = 3.0
KD_WEIGHT = 0.5
EXPERIMENTS = [
    {"num": 1, "name": "lora", "method": "lora", "rank": RANK, "alpha": ALPHA, "lr": LR},
    {"num": 2, "name": "lorma", "method": "lorma", "rank": LORMA_RANK, "alpha": LORMA_ALPHA, "lr": LR},
    {"num": 3, "name": "full", "method": "full", "rank": None, "alpha": None, "lr": FULL_LR},
    {"num": 4, "name": "distillation", "method": "lora", "rank": STUDENT_RANK, "alpha": STUDENT_ALPHA, "lr": LR, "kd": True},
]

# None = look for them in /kaggle/input
CKPT_PATH = None
TOK_PATH = None

# not stored in the checkpoint, same as phase 1
MAX_SEQ_LEN = 1023
ATTN_WINDOW, GLOBAL_EVERY = 256, 4
# the classifier runs every layer with this causal sliding window
WINDOW_SIZE = 256

BASE_DIR = "/kaggle/working/phase3_rr"
os.makedirs(BASE_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev, "|", torch.cuda.get_device_name(0) if dev.type == "cuda" else "")


## Phase 1 checkpoint and tokenizer

In [ ]:
# take the checkpoint with the highest step if there are several
if CKPT_PATH is None:
    hits = glob.glob("/kaggle/input/**/gqa_swa_best.pt", recursive=True)
    print("checkpoints found:", hits)
    if hits:
        CKPT_PATH = hits[0] if len(hits) == 1 else max(hits, key=lambda p: torch.load(p, map_location="cpu", weights_only=False).get("step", 0))
assert CKPT_PATH and os.path.isfile(CKPT_PATH), "gqa_swa_best.pt not found in /kaggle/input"

# tokenizer that sits closest to the checkpoint in the folder tree
if TOK_PATH is None:
    toks = glob.glob("/kaggle/input/**/bpe_legal_16k_v2.pkl", recursive=True)
    print("tokenizers found:", toks)
    if toks:
        TOK_PATH = max(toks, key=lambda t: len(os.path.commonpath([CKPT_PATH, t])))
assert TOK_PATH and os.path.isfile(TOK_PATH), "bpe_legal_16k_v2.pkl not found in /kaggle/input"
print("checkpoint:", CKPT_PATH)
print("tokenizer :", TOK_PATH)


class CustomBPETokenizer:
    def __init__(self, vocab_size=16384):
        self.target_vocab_size = vocab_size
        self.special_tokens = ["<pad>", "<unk>", "<s>", "</s>"]
        self.pad_token_id = 0
        self.unk_token_id = 1
        self.bos_token_id = 2
        self.eos_token_id = 3
        self.pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\w+| ?\S|\s+(?!\S)|\s+""")
        self.merges = {}
        self.vocab = {}
        self.inverse_vocab = {}
        self._word_cache = {}

    def _tokenize_word(self, word):
        word_tuple = tuple(word)
        if len(word_tuple) <= 1:
            return list(word_tuple)
        while True:
            pairs = [(word_tuple[i], word_tuple[i + 1]) for i in range(len(word_tuple) - 1)]
            valid_merges = [p for p in pairs if p in self.merges]
            if not valid_merges:
                break
            best_pair = min(valid_merges, key=lambda p: self.vocab[self.merges[p]])
            first, second = best_pair
            merged_tok = self.merges[best_pair]
            new_tuple = []
            i = 0
            while i < len(word_tuple):
                if i < len(word_tuple) - 1 and word_tuple[i] == first and word_tuple[i + 1] == second:
                    new_tuple.append(merged_tok)
                    i += 2
                else:
                    new_tuple.append(word_tuple[i])
                    i += 1
            word_tuple = tuple(new_tuple)
            if len(word_tuple) == 1:
                break
        return list(word_tuple)

    def encode(self, text, add_special_tokens=False):
        if not text:
            return []
        cache = self._word_cache
        ids = []
        for word in self.pat.findall(text):
            word_ids = cache.get(word)
            if word_ids is None:
                word_ids = [self.vocab.get(tok, self.unk_token_id) for tok in self._tokenize_word(word)]
                if len(cache) < 2_000_000:
                    cache[word] = word_ids
            ids.extend(word_ids)
        if add_special_tokens:
            ids = [self.bos_token_id] + ids + [self.eos_token_id]
        return ids

    @property
    def vocab_size(self):
        return len(self.vocab)

    @classmethod
    def load(cls, filepath):
        with open(filepath, "rb") as f:
            data = pickle.load(f)
        tok = cls(vocab_size=data["target_vocab_size"])
        tok.vocab = data["vocab"]
        tok.inverse_vocab = data["inverse_vocab"]
        tok.merges = data["merges"]
        tok.special_tokens = data["special_tokens"]
        print(f"Loaded tokenizer from {filepath} (Vocab size: {tok.vocab_size})")
        return tok


tokenizer = CustomBPETokenizer.load(TOK_PATH)
VOCAB_SIZE = tokenizer.vocab_size


## Data

In [ ]:
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
P3 = "Phase_3_RR"


def fetch(rel):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=rel, repo_type="dataset", token=HF_TOKEN)


def read_json(path):
    with open(path, "r", encoding="utf-8") as fh:
        return json.load(fh)


cl_train = read_json(fetch(f"{P3}/Train/CL_train.json"))
cl_dev = read_json(fetch(f"{P3}/Train/CL_dev.json"))
it_train = read_json(fetch(f"{P3}/Train/IT_train.json"))
it_dev = read_json(fetch(f"{P3}/Train/IT_dev.json"))
cl_test = read_json(fetch(f"{P3}/Test/CL_test.json"))
it_test = read_json(fetch(f"{P3}/Test/IT_test.json"))
labels_list = read_json(fetch(f"{P3}/Train/label_vocab.json"))

# label name <-> id
label2id = {label: idx for idx, label in enumerate(labels_list)}
id2label = {idx: label for label, idx in label2id.items()}
NUM_CLASSES = len(label2id)

print(f"Loaded {NUM_CLASSES} classes. Train docs: {len(cl_train) + len(it_train)}. Dev docs: {len(cl_dev) + len(it_dev)}")


## Context windows

In [ ]:
# every sentence becomes [prev] </s> <s> [target] </s> <s> [next]
def build_context_instances(data_dict, has_labels=True):
    instances = []
    class_counts = Counter()

    for doc_id, doc_data in data_dict.items():
        sentences = doc_data["text"] if isinstance(doc_data, dict) else doc_data
        labels = doc_data.get("labels", []) if isinstance(doc_data, dict) else []
        total_sents = len(sentences)

        # encode each sentence once, the neighbours reuse it
        encoded_sents = [tokenizer.encode(s) for s in sentences]

        for i, target_ids in enumerate(encoded_sents):
            prev_ids = encoded_sents[i - 1] if i > 0 else []
            next_ids = encoded_sents[i + 1] if i < len(encoded_sents) - 1 else []

            # prev + target + next with eos/bos between them
            seq = []
            if prev_ids:
                seq.extend(prev_ids + [tokenizer.eos_token_id, tokenizer.bos_token_id])
            seq.extend(target_ids)
            if next_ids:
                seq.extend([tokenizer.eos_token_id, tokenizer.bos_token_id] + next_ids)

            # cut to MAX_LEN so it fits in gpu memory with batching
            if len(seq) > MAX_LEN:
                seq = seq[:MAX_LEN]

            inst = {"doc_id": doc_id, "sent_idx": i, "total_sents": total_sents, "input_ids": seq}
            if has_labels:
                lab_id = label2id[labels[i]]
                inst["label_id"] = lab_id
                class_counts[lab_id] += 1
            instances.append(inst)

    return instances, class_counts


print("Building train data...")
train_data_raw = {**cl_train, **it_train}
train_instances, train_counts = build_context_instances(train_data_raw, has_labels=True)

print("Building dev and test data...")
dev_data_raw = {**cl_dev, **it_dev}
dev_instances, _ = build_context_instances(dev_data_raw, has_labels=True)
cl_test_instances, _ = build_context_instances(cl_test, has_labels=False)
it_test_instances, _ = build_context_instances(it_test, has_labels=False)

# class weights for the weighted ce loss, rarer labels weigh more
total_train = sum(train_counts.values())
class_weights = torch.tensor([total_train / train_counts.get(i, 1) for i in range(NUM_CLASSES)], dtype=torch.float32)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES
class_weights = class_weights.to(dev)


class RRDataset(Dataset):
    def __init__(self, instances):
        self.instances = instances

    def __len__(self):
        return len(self.instances)

    def __getitem__(self, idx):
        return self.instances[idx]


def collate_fn(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    # pad with id 0 and mask False
    input_ids = torch.full((len(batch), max_len), tokenizer.pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((len(batch), max_len), dtype=torch.bool)

    for i, x in enumerate(batch):
        seq_len = len(x["input_ids"])
        input_ids[i, :seq_len] = torch.tensor(x["input_ids"])
        attention_mask[i, :seq_len] = True

    labels = torch.tensor([x["label_id"] for x in batch], dtype=torch.long) if "label_id" in batch[0] else None
    meta = [{"doc_id": x["doc_id"], "sent_idx": x["sent_idx"], "total_sents": x["total_sents"]} for x in batch]
    return input_ids.to(dev), attention_mask.to(dev), labels.to(dev) if labels is not None else None, meta


train_loader = DataLoader(RRDataset(train_instances), batch_size=TRAIN_BATCH, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(RRDataset(dev_instances), batch_size=EVAL_BATCH, shuffle=False, collate_fn=collate_fn)
cl_test_loader = DataLoader(RRDataset(cl_test_instances), batch_size=EVAL_BATCH, shuffle=False, collate_fn=collate_fn)
it_test_loader = DataLoader(RRDataset(it_test_instances), batch_size=EVAL_BATCH, shuffle=False, collate_fn=collate_fn)
print("train", len(train_instances), "| dev", len(dev_instances), "| test", len(cl_test_instances) + len(it_test_instances))


## Phase 1 model

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        variance = x.pow(2).mean(-1, keepdim=True)
        return self.weight * (x * torch.rsqrt(variance + self.eps))


class SwiGLU(nn.Module):
    def __init__(self, in_features, hidden_features):
        super().__init__()
        self.w1 = nn.Linear(in_features, hidden_features, bias=False)
        self.w2 = nn.Linear(in_features, hidden_features, bias=False)
        self.w3 = nn.Linear(hidden_features, in_features, bias=False)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))


_ROPE = {}


def _rope_tables(head_dim, device, max_len=2048):
    key = (head_dim, str(device))
    if key not in _ROPE:
        # rotary position tables
        inv_freq = 1.0 / (10000 ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
        t = torch.arange(max_len, device=device).float()
        emb = torch.cat([torch.outer(t, inv_freq)] * 2, dim=-1)
        _ROPE[key] = (emb.cos(), emb.sin())
    return _ROPE[key]


def rotate_half(x):
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(q, k):
    T, hd = q.size(-2), q.size(-1)
    cos, sin = _rope_tables(hd, q.device)
    cos = cos[:T].to(q.dtype)[None, None]
    sin = sin[:T].to(q.dtype)[None, None]
    return q * cos + rotate_half(q) * sin, k * cos + rotate_half(k) * sin


_MASK = {}


def causal_window_mask(T, window, device):
    key = (T, window, str(device))
    m = _MASK.get(key)
    if m is None:
        _MASK.clear()
        i = torch.arange(T, device=device)[:, None]
        j = torch.arange(T, device=device)[None, :]
        # look back only, and only `window` tokens
        m = (j <= i) & ((i - j) < window)
        _MASK[key] = m
    return m


def attend(q, k, v, window=None):
    T = q.size(-2)
    # global layer or short input: plain causal attention
    if window is None or window >= T:
        return F.scaled_dot_product_attention(q, k, v, is_causal=True)
    return F.scaled_dot_product_attention(q, k, v, attn_mask=causal_window_mask(T, window, q.device))


def init_weights(model, n_layers):
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
    model.apply(_init)
    # smaller init for the output projections
    for name, p in model.named_parameters():
        if name.endswith("o_proj.weight") or name.endswith("w3.weight"):
            nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layers))


# fewer k/v heads than q heads (grouped query)
class GQACausalAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, window=None):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.num_queries_per_kv = n_heads // n_kv_heads
        self.window = window
        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, k)
        # repeat k and v so every group of q heads gets a copy
        k = k.repeat_interleave(self.num_queries_per_kv, dim=1)
        v = v.repeat_interleave(self.num_queries_per_kv, dim=1)
        y = attend(q, k, v, self.window)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(y)


class _CausalLMBase(nn.Module):
    def __init__(self, vocab_size, d_model, layers, max_seq_len):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.use_grad_ckpt = True
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList(layers)
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        # tied weights
        self.lm_head.weight = self.tok_emb.weight
        init_weights(self, len(layers))

    def forward(self, idx, labels=None):
        x = self.tok_emb(idx)
        for layer in self.layers:
            if self.use_grad_ckpt and self.training:
                x = grad_ckpt(layer, x, use_reentrant=False)
            else:
                x = layer(x)
        x = self.final_norm(x)
        if labels is None:
            x = x[:, -1:, :]
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.reshape(-1))
        return SimpleNamespace(logits=logits, loss=loss)


class EfficientDecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, d_ff, window=None):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = GQACausalAttention(d_model, n_heads, n_kv_heads, window)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.attn_norm(x))
        return x + self.ffn(self.ffn_norm(x))


class LegalEfficientTransformer(_CausalLMBase):
    def __init__(self, vocab_size, d_model=1024, n_heads=16, n_kv_heads=4, n_layers=16, d_ff=2816,
                 window=ATTN_WINDOW, global_every=4, max_seq_len=MAX_SEQ_LEN):
        # every global_every-th layer is global (window None), the rest use the sliding window
        layers = [EfficientDecoderBlock(d_model, n_heads, n_kv_heads, d_ff,
                                        window=None if (i + 1) % global_every == 0 else window)
                  for i in range(n_layers)]
        super().__init__(vocab_size, d_model, layers, max_seq_len)


# build the phase 1 model from the config saved in the checkpoint
base_ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
cfg_ = dict(base_ck["config"])
base = LegalEfficientTransformer(**cfg_, window=ATTN_WINDOW, global_every=GLOBAL_EVERY)
# remove the torch.compile prefix if it is there
state = {(k[len("_orig_mod."):] if k.startswith("_orig_mod.") else k): v for k, v in base_ck["state_dict"].items()}
missing, unexpected = base.load_state_dict(state, strict=False)
# lm_head is tied to tok_emb so it is not in the file
missing = [k for k in missing if k != "lm_head.weight"]
assert not missing and not unexpected, f"checkpoint mismatch: missing={missing[:5]} unexpected={unexpected[:5]}"
print(f"loaded {CKPT_PATH} (step {base_ck.get('step')}, val loss {base_ck.get('val_loss')})")
print("config:", cfg_)
del base_ck
# the classifier handles checkpointing itself (USE_GC)
base.use_grad_ckpt = False
base = base.to(dev).eval()
# sizes come from the loaded model
D_MODEL, N_LAYERS = base.tok_emb.weight.shape[1], len(base.layers)
assert base.tok_emb.weight.shape[0] == VOCAB_SIZE, "tokenizer vocab size does not match the checkpoint"
assert MAX_LEN <= base.max_seq_len + 1
print("layer windows:", [l.attn.window for l in base.layers])


# a clean copy of the phase 1 weights for every experiment
def fresh_base():
    m = LegalEfficientTransformer(**cfg_, window=ATTN_WINDOW, global_every=GLOBAL_EVERY)
    m.load_state_dict(state, strict=False)
    m.use_grad_ckpt = False
    return m.to(dev).eval()


## Classifier

In [ ]:
# one phase 1 block but with our own mask, so every layer can share the same one
def block_forward(blk, x, mask):
    a = blk.attn
    h = blk.attn_norm(x)
    B, T, C = h.shape
    q = a.q_proj(h).view(B, T, a.n_heads, a.head_dim).transpose(1, 2)
    k = a.k_proj(h).view(B, T, a.n_kv_heads, a.head_dim).transpose(1, 2)
    v = a.v_proj(h).view(B, T, a.n_kv_heads, a.head_dim).transpose(1, 2)
    q, k = apply_rope(q, k)
    k = k.repeat_interleave(a.num_queries_per_kv, dim=1)
    v = v.repeat_interleave(a.num_queries_per_kv, dim=1)
    out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
    out = out.transpose(1, 2).contiguous().view(B, T, C)
    # residual after attention, then the ffn
    x = x + a.o_proj(out)
    return x + blk.ffn(blk.ffn_norm(x))


# frozen W plus a small trainable B @ A
class LoRALinear(nn.Module):
    def __init__(self, base_linear, rank, alpha):
        super().__init__()
        self.base = base_linear
        out_f, in_f = base_linear.weight.shape
        # B starts at zero so at the start it behaves exactly like phase 1
        self.A = nn.Parameter(torch.randn(rank, in_f) * 0.02)
        self.B = nn.Parameter(torch.zeros(out_f, rank))
        self.scale = alpha / rank
        self.base.weight.requires_grad = False

    def forward(self, x):
        # W x + scale * B A x
        return self.base(x) + self.scale * (x @ self.A.t() @ self.B.t())


# same idea but the update acts on the output of W instead of the input
class LoRMALinear(nn.Module):
    def __init__(self, base_linear, rank, alpha):
        super().__init__()
        self.base = base_linear
        out_f = base_linear.weight.shape[0]
        self.A = nn.Parameter(torch.randn(rank, out_f) * 0.02)
        self.B = nn.Parameter(torch.zeros(out_f, rank))
        self.scale = alpha / rank
        self.base.weight.requires_grad = False

    def forward(self, x):
        h = self.base(x)
        return h + self.scale * (h @ self.A.t() @ self.B.t())


# freeze the phase 1 weights and wrap the target linears (all weights stay trainable for full)
def inject_adapters(model, method, rank=8, alpha=16.0):
    for param in model.parameters():
        param.requires_grad = False

    if method == "full":
        for param in model.parameters():
            param.requires_grad = True
        return model

    for blk in model.layers:
        for parent in (blk.attn, blk.ffn):
            for name, child in list(parent.named_children()):
                if not isinstance(child, nn.Linear):
                    continue

                if method == "lorma" and name in ["q_proj", "k_proj", "v_proj", "o_proj"]:
                    setattr(parent, name, LoRMALinear(child, rank, alpha))
                elif method == "lora" and name in ["q_proj", "v_proj", "w1", "w2", "w3"]:
                    setattr(parent, name, LoRALinear(child, rank, alpha))

    return model


class LegalRhetoricalRoleModel(nn.Module):
    def __init__(self, backbone, num_classes, use_gc):
        super().__init__()
        self.backbone = backbone
        self.use_gc = use_gc
        self.classifier = nn.Linear(D_MODEL, num_classes)
        self.dropout = nn.Dropout(0.1)

        nn.init.normal_(self.classifier.weight, std=0.02)
        nn.init.zeros_(self.classifier.bias)
        self.classifier.weight.requires_grad = True
        self.classifier.bias.requires_grad = True

    def forward(self, input_ids, attention_mask):
        B, T = input_ids.size()
        x = self.backbone.tok_emb(input_ids)
        # every layer uses the same causal sliding window mask
        mask = causal_window_mask(T, WINDOW_SIZE, x.device)

        for blk in self.backbone.layers:
            if self.training and self.use_gc:
                x = grad_ckpt(block_forward, blk, x, mask, use_reentrant=False)
            else:
                x = block_forward(blk, x, mask)
        x = self.backbone.final_norm(x)

        # mean pooling over the real tokens only
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(x.size()).float()
        sum_embeddings = torch.sum(x * input_mask_expanded, dim=1)
        sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
        pooled_output = sum_embeddings / sum_mask

        return self.classifier(self.dropout(pooled_output))


def make_model(method, rank=None, alpha=None):
    bb = inject_adapters(fresh_base(), method, rank, alpha)
    model = LegalRhetoricalRoleModel(bb, NUM_CLASSES, USE_GC).to(dev)

    tr_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot_params = sum(p.numel() for p in model.parameters())
    print(f"[{method.upper()}] trainable {tr_params / 1e6:.2f}M / total {tot_params / 1e6:.2f}M ({tr_params / tot_params:.1%})")
    return model


In [ ]:
# check block_forward matches the phase 1 forward and the lm loss is sane
sample = " ".join(next(iter(train_data_raw.values()))["text"])
ids0 = torch.tensor([tokenizer.encode(sample)[:MAX_LEN]], device=dev)
with torch.no_grad(), torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=dev.type == "cuda"):
    T0 = ids0.size(1)
    _local = causal_window_mask(T0, ATTN_WINDOW, dev)
    _global = causal_window_mask(T0, T0 + 1, dev)
    x_ = base.tok_emb(ids0)
    x_ref = x_
    for blk in base.layers:
        # our pass vs the original block
        x_ = block_forward(blk, x_, _local if blk.attn.window is not None else _global)
        x_ref = blk(x_ref)
    lg = base.lm_head(base.final_norm(x_)).float()
    lm_loss = F.cross_entropy(lg[0, :-1], ids0[0, 1:]).item()
    max_diff = (x_.float() - x_ref.float()).abs().max().item()
print(f"max diff vs phase 1 forward: {max_diff:.4f}")
print(f"lm loss on a train doc: {lm_loss:.3f}")
assert max_diff < 0.5
assert lm_loss < 6.0, "lm loss too high, weights and tokenizer probably do not match"
del _local, _global, x_, x_ref, lg


## Evaluation and submission

In [ ]:
# macro f1 / precision / recall on the dev set
@torch.no_grad()
def evaluate_dev(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for input_ids, mask, labels, _ in loader:
        logits = model(input_ids, mask)
        preds = torch.argmax(logits, dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    mac_f1 = f1_score(all_labels, all_preds, average="macro")
    mac_p = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    mac_r = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    return mac_f1, mac_p, mac_r


# one label per sentence for every test doc, zipped as prediction.json
@torch.no_grad()
def generate_submission(model, exp_num, method_name):
    model.eval()
    predictions = defaultdict(dict)
    test_doc_lengths = {}

    for loader in [cl_test_loader, it_test_loader]:
        for input_ids, mask, _, meta in loader:
            logits = model(input_ids, mask)
            preds = torch.argmax(logits, dim=-1).cpu().numpy()

            for p, m in zip(preds, meta):
                doc_id = m["doc_id"]
                sent_idx = m["sent_idx"]
                predictions[doc_id][sent_idx] = id2label[p]
                test_doc_lengths[doc_id] = m["total_sents"]

    final_output = {}
    total_sents = 0

    for doc_id, expected_len in test_doc_lengths.items():
        # rebuild the doc in sentence order
        doc_preds = [predictions[doc_id][i] for i in range(expected_len)]
        assert len(doc_preds) == expected_len, f"Doc {doc_id} length mismatch!"
        final_output[doc_id] = doc_preds
        total_sents += expected_len

    pred_path = os.path.join(BASE_DIR, f"prediction_{method_name}.json")
    zip_name = f"/kaggle/working/submission_exp{exp_num}_{method_name}.zip"

    with open(pred_path, "w", encoding="utf-8") as f:
        json.dump(final_output, f, indent=2, ensure_ascii=False)

    # prediction.json has to be at the root of the zip
    with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(pred_path, arcname="prediction.json")

    print(f"wrote {zip_name} | {len(final_output)} docs, {total_sents} sentences")


## Training loop

In [ ]:
# hard ce loss plus the temperature kl against the teacher
def kd_loss_fn(student_logits, teacher_logits, true_labels, T=2.0, alpha=0.5):
    ce_loss = F.cross_entropy(student_logits, true_labels, weight=class_weights)
    student_soft = F.log_softmax(student_logits / T, dim=-1)
    teacher_soft = F.softmax(teacher_logits / T, dim=-1)
    kl_div = F.kl_div(student_soft, teacher_soft, reduction="batchmean") * (T ** 2)
    return (alpha * ce_loss) + ((1 - alpha) * kl_div)


# trains for a fixed number of epochs, prints dev scores every epoch
# teacher given = distillation, save_best_to given = keep the best dev epoch on disk (used as the teacher later)
def run_training(model, opt, scaler, tag, epochs, teacher=None, save_best_to=None):
    best_f1, t0 = 0.0, time.time()
    for ep in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        bar = tqdm(train_loader, desc=f"{tag} ep{ep}/{epochs}", leave=False)

        for input_ids, mask, labels, _ in bar:
            if teacher is not None:
                with torch.no_grad(), torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
                    t_logits = teacher(input_ids, mask)

            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
                logits = model(input_ids, mask)
                if teacher is not None:
                    loss = kd_loss_fn(logits, t_logits, labels, T=KD_TEMP, alpha=KD_WEIGHT)
                else:
                    loss = F.cross_entropy(logits, labels, weight=class_weights)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            train_loss += loss.item()
            bar.set_postfix(loss=f"{train_loss / (bar.n + 1):.4f}")

        f1, p, r = evaluate_dev(model, dev_loader)
        print(f"  [{tag}] ep {ep}: loss {train_loss / len(train_loader):.4f} | dev macro P {p:.4f} R {r:.4f} F1 {f1:.4f} | {(time.time() - t0) / 60:.1f} min")

        if f1 > best_f1:
            best_f1 = f1
            if save_best_to:
                torch.save(model.state_dict(), save_best_to)
    return best_f1


## Train

In [ ]:
del base
gc.collect(); torch.cuda.empty_cache()

use_amp = dev.type == "cuda"
teacher_file = os.path.join(BASE_DIR, "teacher_model.pt")

for exp in EXPERIMENTS:
    print(f"\n===== exp {exp['num']}: {exp['name']} =====")

    teacher = None
    if exp.get("kd"):
        # the teacher is the best dev epoch of the full fine-tuning run
        teacher = make_model("full")
        teacher.load_state_dict(torch.load(teacher_file, map_location=dev))
        teacher.eval()

    model = make_model(exp["method"], exp["rank"], exp["alpha"])
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=exp["lr"], weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    run_training(model, optimizer, scaler, exp["name"], EPOCHS, teacher=teacher,
                 save_best_to=teacher_file if exp["method"] == "full" else None)
    generate_submission(model, exp["num"], exp["name"])

    del model, optimizer, scaler, teacher
    gc.collect(); torch.cuda.empty_cache()
